# 다음 위치 도구: IG 기여도 → 여러 부위 후보 → 실제 9칸 평가

**파일: `lvlm_missing_ig_location_dev_colab.ipynb`**

**L4/A100 → 기본값 유지 → 모두 실행 → Drive 연결 허용. 기능 제작 완료, 실제 위치 개선 효과는 실행 전입니다.**
이전 코랩 재실행/추가 표시/토큰 입력은 불필요합니다. supervised 원본 ZIP 한 개를 자동 검색합니다.

- 이전 실제 PR68의 누락 유무와 모든240답을 고정하고, 판별기 양성26장에만 새 위치 도구를 적용합니다. 누락25/25 검출은 고정 입력이며 이번 도구의 새 개선으로 주장하지 않습니다.
- 같은 동결 DINO+분류기의 점수에 기여한 픽셀을 Integrated Gradients로 계산합니다. 흐린 사진/전체 평균색 두 기준을 쓰며, 각32단계입니다. **역전파는 해석 계산이며 가중치 학습/변경은 없습니다.**
- 기존의 큰칸 가림 top1과 IG top1/여러부위를 비교합니다. IG 여러부위와 같은 수의 기존가림순위/중앙/해시무작위 칸 대조도 비교합니다. 단순히 칸만 늘린 효과를 구분합니다.
- 원본 좌표로 복원하며 정답 위치24장/판단불가1장은 채점에만 씁니다. 위치를 보류해도 누락 유무는 바꾸지 않습니다.
- 기존 특징2개를 재계산해 동등성 검사 후26장 각각의 원본특징도 비교합니다. 최대 약1744 forward/1664 backward, DINO만 사용합니다. 새 LVLM/재학습/Q1Q2gold0.
- IG는 판별기 설명이지 결함 위치 정답이나 분할 모델이 아닙니다. 배경에 반응할 수 있고 두 기준에 민감할 수 있습니다. completeness 수치오차와 원시 기여도도 저장합니다. 효과는 실행 후 확인합니다.
- 기존 dev를 반복 탐색한 실험입니다. 최종 Agent/새 holdout 성능이 아닙니다. 정상 유무 오탐은 고정되어 있으므로 개선됐다고 해석하지 않습니다.
- 이미지별로 별도 Drive 폴더에 저장합니다. 같은 입력/코드/환경은 성공 이미지부터 재개하고 실패만 재시도합니다. 결과 ZIP은 기여도 맵을 포함해 수십 MB일 수 있습니다.

In [ ]:
RUN_FOLDER = "lvlm_insplad_results/missing_ig_location_dev240_v1"
SUP_ZIP_PATH = ""

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','numpy==2.1.3','pillow==11.3.0',
    'transformers==5.14.0','accelerate==1.14.0','timm==1.0.30'],check=True)
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory<20*1024**3:
    raise RuntimeError('L4/A100 VRAM20GB 이상을 선택하세요.')

In [ ]:
import base64,hashlib,io,json,zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
DRIVE=Path('/content/drive/MyDrive')
SOURCE_SHA256='66e7e6f5209af548428653e5c0c1a70c1eb4aa4a883f3e07704265bca1d34834'
SOURCE_MANIFEST={'inputs/dataset.json': '301011c1b9e7a4c7f36e1cd1385587b6d0dfa09ff4d4cdc514266cc807113a99', 'inputs/head.json': 'd9ff0610d0b74752a2dfc5b53f1cdc1db5e916e17b4260172f1c7abab8c6e7f1', 'inputs/baseline.json': '1e516f5c186cabac61be6bc9a60baca9b32cacfc525c34892248b7881b0d8b97', 'inputs/tools.json': '072364b7a0da783c8f9cb4f79806bf5964fbfcd527276964ce749195a281c5d8', 'inputs/old_maps.json': '01a9b561a7802863e0683346aecf955d93e2776150fd568b276fd28fb74560f3', 'inputs/evaluation_gt.json': 'b784dc8c9fedbd4db429176cf662f77eeecca538d2fa5462c463ad84356cb9ab', 'inputs/annotations.json': '9a9e71b995174288e3507e0d60f97fd155abb2149de2bf4ee0feb10fe02d52d3', 'inputs/features/485839154ffbf0a67d72019a1cc0f1f01efb9ce7d06dc5f6f4c10bc00f2513e8.npy': 'cb8103b0c630d00808dbd3d66c825fdf37a7b63a20f2e92d280926aff492eb47', 'inputs/features/2879741287e867404f07a7ba2535001e91fe3b2cf4e5eb593565635d30603e76.npy': 'af04e02d2cc57392189c157c47a6bbdb50abc43d966ec69b4839e52cff50db42', 'inputs/features/a38e6f97d6cfa7f7fc558424715dedc33163c67f9f9585ad4d418ab2387c3a5a.npy': 'bde9f5f8d05faee8cf5c962876d839eb90f38c2414313cbb24c4ffbd1c69bc32', 'inputs/features/a77c71974bcf38257567c87b9c9b4675b66bf88ac3ba0be537cbbfe99e3643de.npy': '47387429c26ac6651df865d70277459d2b3ca7b7442ff18bd7dd42fed9c4615f', 'inputs/features/432592ac6781d53c646941d8ae1ddc72755afb8eda9eb44336b366c171acf992.npy': '061371067bcd44ee78037cc74382db23a7a15ba49e6076ec4fe7dfd6a89b3af3', 'inputs/features/957c31a30fa796851f3206e5671d526d9f716a281ea79c186bded7f1e07ab408.npy': '1d6c094822ec8fa29e48d1786e33db61cd05faa49ad148c11cb5465b2b2c3cf1', 'inputs/features/1fe146f99c37584c2173756e6ede128385076d5ea4dd3fbdd47e3bab05d49113.npy': '2f236c4400b65870fb4e183faa4976b48e952478694b81c6202183d3aecf82ce', 'inputs/features/ffd421df7e1fe9ad7e299b3258ca3cc954b054c18c53964d5154b6f88058f3e5.npy': 'b96d1858b2ff4da497d831f165e991badbfe53595768881a3aefdcac56adf95d', 'inputs/features/10e15ebe0c3a5166dacf3f1318d3a8dbfbe84690669bd82af25862e25fd5f785.npy': '6893218fefd871af4e80f0d055d3fa9432d53d283de074d91ee571f9618484a8', 'inputs/features/74b7f98c5dd7b9f12ac193db0cd2eef616cd7700bcb44763d1111b14435e4f44.npy': 'cc4bc97d4f4d02df49791e6a3d8f8694fe65fcf6a2af2e967fc1338210341b2c', 'inputs/features/b6133af1f9d13e0ca759948a5f66d4be5e4b52d9fb70c8fc91f4c451f5f852db.npy': '9b4d0a48272e4c7cf1ec8cce300d6a8b0866e85911516826342403d4976e5d68', 'inputs/features/57dd4484b64613603a2377522cf1dc198c00d04fa7b7c4e9571d48d07af4571a.npy': '8145954e807bc8383c7cbc4164a4d79e97b6217f1d8d420ab751045cede4638d', 'inputs/features/6f76a9f359f63375beb6d8060d2af5559f96b7eaa56f0c617599eadf847d3da9.npy': '21a766daf2ec4930c912581fe836c6c494183e525c29601e67e5d950e6272d46', 'inputs/features/5ec0a94f488133a6c240845fb494083211c7eee367eea5db3eb05b897775411a.npy': '2c200374dba391decc7367a2a7e64876f93325cbafb510ee1f0e9e6486580656', 'inputs/features/76567a0f05018c1b6bcba032be1d90c9f64d5f669ccd60a56982ad7fb2217a6f.npy': 'e6791f7fac0207712ade688eee98c1464d5ee04f46007a3ad53de3aff2833ad5', 'inputs/features/34847c57f5abb74e32e8fba7ceb0e43b2d67248a69004262c41590a3dd4d8443.npy': '012fed5b48423710b1469b5f5fa2d12490f998bc805e9844c613ee15f30d5cd3', 'inputs/features/9860398a1f40106b82e79e19f35181e3806ee5468e158d25f504f8256c64423f.npy': 'e173a8c0b85851f3484cedf0436d7026cd766bbed3a9deb6adc0491fb57e9c71', 'inputs/features/59b8fe68eb75205c8c98537a8ceda303d3fbc730858570b5c252caf253c5f20a.npy': '43d6de70ee11a9228129573054f6611e76bf668d19ed86861f7dff26134f90c3', 'inputs/features/5ed0553b85d9fe2356bb7a31e35d683fc0668c449e2e1e97807cbcf6f727ab50.npy': '436bb758dafe12333df5a817437ee58037dcd5b881edc369373803fd8124773f', 'inputs/features/9d2641158d7ab777a9c95edce4dcabb0e45c9013d575a891436fa4a59bbb2612.npy': '056741b98f8abaa34a7a57d4d183a714b471b146f208bf1ddadbf5d82bf54763', 'inputs/features/01871b3115e2cea822716fcb1459e7ede54894ea4442de9ee65205e82455ecc5.npy': '5cdef91629b55d0f5e7c832802cec1a92dc40e32bd5c8c92be14608e46a657dd', 'inputs/features/0a3759914ac2afcb7a63a09c36c0eecad70ce3b1ce23fa50f03c00a3d05f0f69.npy': 'fab88bd2b8f3be42d212f60c0e363bf5287ba47bc2b3a6c71a61c0d97b1a6c5a', 'inputs/features/adaf469e13ae944830cfa1be6332829c4d9766fee89ff3e44f29ecd3575d9c8c.npy': '9d69d2f18acbfaee414bd5d60d7dbc69fc1de8383ad9740e34dbca23080433ee', 'inputs/features/349a817ef95a5ab63ab57c4f19484ddfa2e8e34cc6d1429395cc35a657b80e7b.npy': '55c178210ad6596c81d3cf3fe35d9ac61ef64cedfcc8bb886fed5c2ca12e02fb', 'inputs/features/1ededacb49296765623c78588cbb3f0a6a289eb600a51e0671d0422cb506bb68.npy': '5008ed7f9cc7744389093ae4896e042d06dda25c5c5addd212c4ea1058fc54df', 'inputs/features/e3b6462b28aa164d6b2e630d0a3d90e3a7f4776bfd4a2b8508d7aa06cee2719f.npy': 'abe754626c8ecf1206dcb14192e243c1c3e165910736be47081f6b368a4c7232', 'inputs/features/6fa216657434a04c9aeb7774e9ef42961fb3ac323d0b25bcd2b7f69533e7f02d.npy': 'c6b9a40c864647d50fa4ee7e8337c8f7a9219b18b2c9f448e212fb75e49c1597', 'inputs/features/5bc79a18fe166efb14af486fbca0900addfa9542a01c4300329419ff76d3cb84.npy': 'f380f5558ed771b5dc56ff2a48e86acdeda429a496ad0142ebae81699cba3dcf', 'inputs/features/a1e808dbb67b69b94b923b107cbc6639f959189cbc2c99fb44f57c2c47039646.npy': '98d61a31858015ed73e481e2cdf5f92fc158b8a1fe87e1d6937d38f447516717', 'inputs/features/7089559a76ea97b58ef6340b13ccad81786c4716a26bd037f1d0ef174f492e12.npy': '35ae48e6fee00cac11cc40dd7ab956ffc0de22e73407cd03fde3002d5140e2b5', 'inputs/features/4d0dc26be7117351fbb0423e280571a86f6bbf8674f08ed8555c7a213674c5cf.npy': '25932ae0504f78f079d7c01f0bcc3006b14ec82eae43f1bd3bbdb9d3a156469a', 'inputs/features/0930d8dfc1ccbdfdcde628cfd0a21f56152d55e4a67293f706d0c30ff373601d.npy': 'a343b4ff91fe2019e139913bf5d4b6c9f609d7b3d6eac9d83c061f1f2f308388', 'inputs/features/a628d5f1c7169ad835d42723e5a3f72d2e6a3f8eaba0c312be1034956c4e10ea.npy': '143d4216117c70f06753ddddbee27c919f473b187b78e573f51e3b513a95d639', 'inputs/features/33444f41cdde2a718b10620f7188e25fd75179c92bade11ac18123c741503df3.npy': '890f434b1ecadd680677d03ef61482c6ab4df8852fe1a02feaca2fb192198f12', 'inputs/features/b4419670db6b1a0fe43d4beea52e2b4121e44cc43b6efce9c1d7a4b8bf28b257.npy': '3e4f7da9db78186977fc2d6bb84407cc1d9de6f9919ec06f3c17dc6b5e6b80f1', 'inputs/features/fbdc6c092cd469552d6040f3710b845b488d1d6af054533fb99cc4236564373d.npy': '654d6c9a1af27ea197df355affd1be2906afd29d4b39e375a17cae716635ff66', 'inputs/features/c47c161362e17692bba6f16da30db0875c36a477ea95ddc803913c9975f75c8d.npy': 'fbaa634ff28dea2e36185592579bedc8d820381af20fd21e702e325ee4ebb836', 'inputs/features/98f9b7dd2011107ebab68dccdc4da448b8b3c3be73a48582795d0f3f66e4f14b.npy': '4211b19c341659423e830f76f2453024296be7042dce439935820c4ed8281cd2', 'inputs/features/bcd161cbff331dbc010fa1a7dc4e443fe8662416998fcd0e5fc3e6f22b533808.npy': 'b2260bc48e6a0a1a16ad7c1f71c896dd6ba7bc534421c5a6f66169f4ce17316d', 'inputs/features/116476bbe2e98c8f96324d822e3c82addacafa843bfe2171aac708d261d8e618.npy': 'b6274409634f33d401ebdfeff330d4a5b9be10f745bafbe19e94a7488d93df5b', 'inputs/features/fc0410b1e332077756dad84e58532b9237037f6547361ac3799f8d67794a6343.npy': '55bb672eafa817df4723e514d56ac4686bec7fcfcd92e50a8b93cfb07d289250', 'inputs/features/10d5031bb15180b6ac9f875adaaff40a6c9bb5e09dae367097971b349277a165.npy': '0f3255f6574a8ac7dd26c822a85620cfa424bdd669637abea1cd446a79f267dc', 'inputs/features/d1db092b343b5e5a4a15571ad47ffdb4ba2b04b09718f35cce0beeec6e92c69b.npy': 'cf754266e7cb23af95a7994c4359869af876cb34ca380470c6d2e81c0979ef7a', 'inputs/features/df1f72daac6e2399e380a229a9cbf39b1c0dab67e863baaa15c7fc8f21089290.npy': 'ce96eabef7cd121383b4f7c46b2596204f0bc5f0662ecaf9df6d972f185f20b0', 'inputs/features/4d6b3718bebaba57578002f0d74bc12faf79464a2b5449be6947440db21a7848.npy': '8663a76dd624a6aee52cc430e9beae4169a7b4e631b8d674b88bba79aac00fbe', 'inputs/features/06913bc78c099628747cfc2cea5ebfafb6f79fd08155950bb650cb63563f5fbe.npy': '7aed801959b7c6f5258fd49e8c33987f987cd53218cbf864d3fae1a228300fda', 'inputs/features/9331a21c644d527c8eaabc7c055fd127a31036e5da5411d057d7fdf38bc56bab.npy': '6047a22ecd4481ac192587dd6c265089f8db1c049d818d207c6c77d9f48b1b27', 'inputs/features/69758e4f18d3f39ce7f3a86bb90a6c958f8cd78882c908fc60b2b7ed8635fd77.npy': '219adfcaeee808421b2f16217e0a5edff154b48d78676851c66aa31f828926d8', 'inputs/features/4e41b01147d9f1c2eb33c9e468879f8dbaedddd88d4a76ae997812326b474a07.npy': '73260de5d115e90ebe8cf66dfe134f273d7d51122cf1617faf228940f4811556', 'inputs/features/7e7f427fbb13c18ce9d92dc886b3c5c24e5bd7972f22fc8dfbb8216afc03b06b.npy': 'aca75fb02e0c127da309726df67b0dfdc96ecac890eb86c22c3bfcb5a64c2f4b', 'inputs/features/57215aef477d29322472fe2877a932e2aa0635941bd29efe36a85aa5b219e872.npy': '9aa740a26e45b734213e13047b0cfe82818ecffe09cf02721d19fb680676d7b4', 'inputs/seeds.json': '1eca4c301da97bd3ad148f809d81860a4670052d0d2476564e7c7b16f7587e10', 'inputs/provenance.json': 'f928ac02117d2ac3f85e15d476f91a6d52898ef85b131e398a32a86887fcf5f9', 'scripts/build_explainable_qa_gold.py': '291f19e6c89abf43890b74cc43643db77b0d1fc0506679e935377b44abaa5cdc', 'scripts/build_gold_v2_dev_set.py': '372907c342c47a38791f44b09710b6c662c5d56760f0f2724c36435d13ffd5bb', 'scripts/build_missing_ig_location_notebook.py': '42c1726d24634392a3fed9d32c2578dae3dc2afe92880764c21cebc751da413c', 'scripts/build_retrieved_refs.py': '2b18d709a5733f07cfbaee370630d46f33fa5afe7ac80ba9005145278cb178fb', 'scripts/dev_part_crop_annotation.py': '841e16078371cf7384c428944b103c864b5be2f8827ba8ca48901aee5bf30744', 'scripts/loc_box_annotation.py': 'bc92ba581d54163019eb5b658ef91007e69164c13495ab34dec382f729340466', 'scripts/missing_ig_location.py': '0627950d8cff09f2a69194919f3b978206e80c6aa12b418c9f4ff7ae23712ce5', 'scripts/q3_multiregion.py': 'c431048fec6e06e6f26e2f57628e74a7861765aa4f0e63905ae16c3b40d12752', 'scripts/run_dev_prompt_diagnostic.py': 'cdca4100de8ca83d65356c2dbde4a527869a2e095957ad6329ac0f879b15cf75', 'scripts/run_gold_baselines.py': 'c5b5507c317654504725cf1c3efa9ad1e09daf00fa62d4331cb228c531a76602', 'scripts/run_gold_v2_baselines.py': '0852ea87d20951713c3b98ca67f88472f41a29481767f8947f25efd6be37f650', 'scripts/run_gold_v2_consistency_metrics.py': '802267807ee16dfa5b700f8676f2d8c294e733c99fc1586884896e164c8b3ec6', 'scripts/run_gold_v2_dev_baselines.py': '6104d142d3f52ab85e6ca130c32a4d6f1a08c7f676e199f003a2a963d9ab51de', 'scripts/run_gold_v2_q23_candidate.py': 'caad191a51a9f24685bc64e60ec407276ddfd191e572c337eb5e8909c9b6710c', 'scripts/run_gold_v2_q23_zoom_prompt_dev.py': 'c6d74f5276c22e74b43c4321acf0a7e93897499ed2346f2ae4f3f6e3302c91cb', 'scripts/run_gold_v2_q2_q3_agent_dev.py': 'd7196adf586797e9131cc52d005d6d34c30ebe3cb751ce3e8e49eade6db2d44a', 'scripts/run_gold_v2_q2_q3_refined_dev.py': 'dcd4ba3d77d2668d85098c05c889e7ea520ccf2ba4a97ac6b47dc9dbe4deafef', 'scripts/run_gold_v2_reference_ablation.py': '25b52f00cbc7911149b2377bdf0129f46cd1575831ac7f8b83dc1665534f8e6d', 'scripts/run_gold_v2_verify_agent.py': 'ded0d022e13ecf64971fd1e5d40e5595f166c1a99277837662de0e4a019126f1', 'scripts/run_missing_component_probe.py': 'b646cd82896c401dc6d756fdbd8e534d393eebacd59c19e409685ae498a2437a', 'scripts/run_missing_expert_direct_dev.py': '2251a245427edffd9cea58ced64b67185d402c8cfb2915e2926b6c33112971fd', 'scripts/run_missing_ig_location_dev.py': 'ae73f6bd349bb3149eb785fb331449d70fcd5b31c8ee7a93896668758559a22f', 'scripts/run_q2_conflict_evidence_dev.py': '7f830216585b43d6118dfe648549cbcce8c1f78028d3675b3437431f8ab36da6', 'scripts/run_q3_missing_occlusion_dev.py': '563c85233ab7a272a5323f12732e8649a50678e7ab73903eab71e30224d9ea21', 'scripts/run_selective_missing_expert_agent_dev.py': '4cf263d5ffe7281237b99c943f2de528db9510a8a236e05017dd62d749dba5e2', 'src/insplad_benchmark/__init__.py': '8f82d2ef2a6c2a1922f1d9e1a947e24796940292bce416100e407d07a5023cad', 'src/insplad_benchmark/dataset.py': '9194614b34c3f556d9b2399b6cc896b991cde37dbb325edc5f582ee9db547874', 'src/insplad_benchmark/hf_adapters.py': '4e5e39fe437a3c7dcb52be598cbc1d1ac9acabb0b2414d8a259173a9b6d577dc', 'src/insplad_benchmark/metrics.py': '27d6b7a34515487b1ed443b671c3bc936305cb790a5b53feb750f39bfb3c3746', 'tests/test_missing_ig_location_dev.py': 'ed99a4a5aec8cf4a42720425cb16e919bb4813901830cab2fc5729d20e337512'}
SOURCE_BYTES=base64.b64decode('...')
assert hashlib.sha256(SOURCE_BYTES).hexdigest()==SOURCE_SHA256
CODE=Path('/content/missing_ig_location_'+SOURCE_SHA256[:12])
with zipfile.ZipFile(io.BytesIO(SOURCE_BYTES)) as archive:
    assert set(archive.namelist())==set(SOURCE_MANIFEST)
    for name in archive.namelist():
        target=CODE/name;content=archive.read(name)
        assert target.resolve().is_relative_to(CODE.resolve())
        assert hashlib.sha256(content).hexdigest()==SOURCE_MANIFEST[name]
        if target.exists() and target.read_bytes()!=content:raise RuntimeError('내장 소스 변경; 새 런타임을 사용하세요.')
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(content)
(CODE/'source_manifest.json').write_text(json.dumps(SOURCE_MANIFEST,indent=2))
OUT=DRIVE/RUN_FOLDER
if not OUT.name.startswith('missing_ig_location_dev'):raise ValueError('별도 실행 폴더 이름을 유지하세요.')
RETURN_CODE=98;LOG=DRIVE/'missing_ig_location_dev240_console.log'
print('SOURCE',SOURCE_SHA256,'26장 위치 도구 실험. 유무 고정/새LVLM0/학습0/IG역전파해석만 수행')

In [ ]:
def find_zip(override):
    name='supervised_fault_classification.zip';size=874753167
    if override:
        path=Path(override)
        if not path.is_file() or path.stat().st_size!=size:raise FileNotFoundError(str(path))
        return path
    for path in [DRIVE/name,DRIVE/'lvlm_insplad_data'/name]:
        if path.is_file() and path.stat().st_size==size:return path
    matches=sorted(p for p in DRIVE.rglob(name) if p.is_file() and p.stat().st_size==size)
    if not matches:raise FileNotFoundError(name)
    return matches[0]
SUP=find_zip(SUP_ZIP_PATH)
command=[sys.executable,str(CODE/'scripts/run_missing_ig_location_dev.py'),
    '--capsule-dir',str(CODE),'--sup-zip',str(SUP),'--out-dir',str(OUT)]
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with LOG.open('a',encoding='utf-8') as log:
    for line in process.stdout:print(line,end='');log.write(line);log.flush()
RETURN_CODE=process.wait();print('RUNNER_RETURN_CODE',RETURN_CODE)

In [ ]:
from google.colab import files
RESULT=DRIVE/'missing_ig_location_dev240_results.zip'
with zipfile.ZipFile(RESULT,'w',zipfile.ZIP_DEFLATED) as archive:
    if OUT.exists():
        for path in sorted(OUT.rglob('*')):
            if path.is_file() and '_images' not in path.relative_to(OUT).parts:
                archive.write(path,'run/'+path.relative_to(OUT).as_posix())
    archive.writestr('source_bundle.zip',SOURCE_BYTES)
    archive.writestr('source_manifest.json',json.dumps(SOURCE_MANIFEST,indent=2))
    archive.writestr('runner_return_code.json',json.dumps({'return_code':RETURN_CODE}))
    if LOG.exists():archive.write(LOG,'console.log')
status=json.loads((OUT/'run_status.json').read_text()) if (OUT/'run_status.json').exists() else {'complete':False}
print('RUN STATUS',status)
if RETURN_CODE or not status.get('complete'):print('실패/미완료입니다. ZIP을 보존해 전달하세요.')
if status.get('completeness_warnings'):print('일부 IG 적분 오차가 큽니다. 원시 맵·진단과 함께 분석해야 합니다.')
if not RETURN_CODE and status.get('complete') and (OUT/'leaderboard.csv').exists():print((OUT/'leaderboard.csv').read_text())
print('ZIP',RESULT,'SHA256',hashlib.sha256(RESULT.read_bytes()).hexdigest())
files.download(str(RESULT))